# Lección 1.5 — El ciclo de vida de un proyecto de ML: de la pregunta de negocio al modelo en producción

> **Módulo 1 · ¿Qué es aprender para una máquina?** · Nivel 1 — Cero
> ⏱️ Tiempo estimado: 75–90 min (lección con ejercicios intercalados ≈ 55 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lecciones 1.2 (vocabulario) y 1.4 (tipos de problema).

### 🎯 Objetivos
Al terminar esta lección podrás:
- Enumerar y explicar las **etapas del ciclo de vida** de un proyecto de ML.
- Recorrer un proyecto completo en miniatura: pregunta de negocio → datos → línea base → modelo → evaluación → puesta en producción → monitoreo.
- Traducir una métrica técnica (error medio) a **dinero** para el negocio.
- Entender por qué un modelo en producción **se degrada** y hay que vigilarlo.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números o palabras.

## ▶️ Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor

plt.rcParams["figure.figsize"] = (8, 3.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: abrir un restaurante, no cocinar un plato

Entrenar un modelo es como cocinar un plato. Pero un **proyecto** de ML se parece más a **abrir un restaurante**: hay que saber qué quiere la clientela, conseguir ingredientes, probar recetas, abrir las puertas… y seguir revisando cada semana que la comida siga saliendo bien.

```
     ┌──────────────────────────────────────────────────────────────┐
     ▼                                                              │
 1. Pregunta    2. Plantear     3. Conseguir    4. Explorar y       │
    de negocio ──► como ML ────► los datos ───► preparar datos      │
                                                     │              │
                                                     ▼              │
 8. Monitorear ◄── 7. Poner en ◄── 6. Evaluar ◄── 5. Línea base     │
    y mantener       producción      (¿sirve al      y modelos      │
       │                             negocio?)                      │
       └──────────────── reentrenar / replantear ───────────────────┘
```

Fíjate en dos cosas: el modelo propiamente dicho es solo **una** de las ocho etapas, y el proceso es un **ciclo**: casi nunca se termina.

### ✋ Ejercicio 1 — ¿Dónde se va el tiempo?
En encuestas a profesionales del ML, ¿en qué etapa crees que dicen pasar **más** tiempo: entrenando modelos (etapa 5) o consiguiendo y preparando datos (etapas 3–4)?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

En **conseguir y preparar datos**. Las encuestas a científicos de datos suelen situarlo en más de la mitad del tiempo de un proyecto; entrenar modelos es una parte mucho menor. Por eso el Módulo 2 entero está dedicado a los datos.
</details>

---
## 2. 📖 Etapa 1 — La pregunta de negocio

Todo empieza con un **problema**, no con un modelo. Nuestro caso:

> ☕ **Cafetería "El Grano".** Cada mañana hornean **120 croissants**. Algunos días sobran (se tiran) y otros se agotan a mediodía (clientes que se van sin comprar). La dueña pregunta: *"¿Cuántos croissants debería hornear cada día?"*

Antes de tocar datos, conviene responder:
- **¿Qué decisión** se tomará con la predicción? → Cuántos croissants hornear cada mañana.
- **¿Cómo se mide el éxito** para el negocio? → Menos dinero perdido (por sobras + ventas perdidas).
- **¿Cómo se hace hoy?** → Siempre 120. Esa será nuestra primera **línea base**.

### ✋ Ejercicio 2 — Malas preguntas de negocio
¿Qué problema tiene cada una de estas "preguntas de negocio"?
1. "Queremos usar inteligencia artificial en la cafetería."
2. "Queremos un modelo con 99 % de exactitud."
3. "Queremos predecir las ventas de croissants del próximo año, día a día, al croissant exacto."

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. No hay **problema** ni **decisión**: es una solución buscando un problema.
2. Es una métrica técnica sin conexión con el negocio (y, para predecir un número, la "exactitud" ni siquiera tiene sentido; Lección 1.4).
3. Es irreal: nadie puede predecir el tiempo o los festivos de dentro de un año al detalle, y la dueña solo necesita decidir **para mañana**. Pide más precisión de la que la decisión necesita.

Una buena pregunta de negocio nombra la **decisión**, el **horizonte** (para mañana) y **cómo se mide** el éxito.
</details>

---
## 3. 📖 Etapa 2 — Plantearlo como problema de ML

| Pregunta | Respuesta para "El Grano" |
|---|---|
| ¿Tipo de aprendizaje? (Lección 1.3) | **Supervisado**: tenemos el historial con las ventas reales |
| ¿Tipo de problema? (Lección 1.4) | **Regresión**: predecir un número (croissants) |
| ¿Qué es un ejemplo? | Un **día** |
| ¿Etiqueta? | Croissants **vendidos** ese día |
| ¿Características? | Lo que se sabe **la mañana anterior**: día de la semana, temperatura prevista, si se espera lluvia, si es festivo |
| ¿Métrica técnica? | **Error medio** en croissants (MAE, Lección 4.1) |
| ¿Métrica de negocio? | **Dinero perdido** por día |

### 🔮 Ejercicio 3 — La trampa de la etiqueta
Los días que se agotaron los croissants, el registro dice "vendidos: 120". ¿Es esa la **demanda real** de ese día? ¿Qué problema crea para el modelo?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

No: si se agotaron, la demanda real fue **120 o más** (quizá 150), pero solo vemos 120. A esto se le llama **demanda censurada**. Si entrenamos con esos datos tal cual, el modelo aprenderá que nunca se venden más de 120 y **subestimará** los días buenos.

Soluciones reales: anotar cuántos clientes pidieron croissant cuando ya no había, o marcar esos días como "agotado" y tratarlos aparte. Pensar **de dónde sale la etiqueta** y si es fiable es parte del trabajo (Lección 1.3). En esta lección supondremos que la cafetería registró la **demanda real** (contando también los pedidos que no pudo atender).
</details>

---
## 4. ▶️ Etapas 3 y 4 — Conseguir, explorar y preparar los datos

"El Grano" tiene un año de registros (365 días).

In [ ]:
rng = np.random.default_rng(5)
dias = 365
cafeteria = pd.DataFrame({"dia_semana": np.arange(dias) % 7})     # 0 = lunes … 6 = domingo
cafeteria["temperatura"] = np.round(20 - 8 * np.cos(2 * np.pi * (np.arange(dias) - 15) / 365) + rng.normal(0, 3, dias))
cafeteria["lluvia"] = (rng.random(dias) < 0.25).astype(int)
cafeteria["festivo"] = 0
cafeteria.loc[rng.choice(dias, 10, replace=False), "festivo"] = 1

extra_por_dia = np.array([0, -5, 0, 5, 15, 40, 30])           # los fines de semana se vende más
cafeteria["demanda"] = np.round(100 + extra_por_dia[cafeteria.dia_semana] - 1.2 * (cafeteria.temperatura - 20)
                                - 10 * cafeteria.lluvia + 25 * cafeteria.festivo + rng.normal(0, 8, dias)).astype(int)
cafeteria.head(8)

### 🔮 Ejercicio 4 — Explora antes de modelar
Antes de ejecutar la celda, apuesta:
1. ¿Qué **día de la semana** se venden más croissants?
2. ¿Se venden más con **frío** o con **calor**?

✍️ *Mis predicciones:*

In [ ]:
nombres_dia = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
cafeteria.groupby("dia_semana")["demanda"].mean().rename(index=dict(enumerate(nombres_dia))).plot.bar(ax=axes[0], color="peru")
axes[0].set_title("Demanda media por día"); axes[0].set_xlabel("")
axes[1].scatter(cafeteria.temperatura, cafeteria.demanda, s=8, alpha=.6, color="peru")
axes[1].set_title("Demanda vs. temperatura"); axes[1].set_xlabel("°C")
plt.tight_layout(); plt.show()

print(f"Demanda media: {cafeteria.demanda.mean():.0f} croissants/día · mínima: {cafeteria.demanda.min()} · máxima: {cafeteria.demanda.max()}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. El **sábado** (≈ 139 croissants), seguido del domingo (≈ 132). El martes es el día más flojo (≈ 92).
2. Se venden más con **frío**: la nube de puntos baja hacia la derecha.

Hornear siempre 120 es claramente mala idea: los sábados se quedan cortos y los martes sobran. **Explorar los datos antes de modelar** te da intuición y te ayuda a detectar errores (Módulo 2).
</details>

### 📖 Dividir: el pasado para aprender, el futuro para examinar
Para saber si el modelo funcionará **en el futuro**, lo entrenaremos con los primeros 300 días y lo examinaremos con los **últimos 65**, que no verá al entrenar. Con datos ordenados en el tiempo, el examen debe ser **posterior** al entrenamiento: en la vida real nunca tendrás datos del futuro para entrenar (Lecciones 2.5 y 4.6).

In [ ]:
caracteristicas = ["dia_semana", "temperatura", "lluvia", "festivo"]
entrenamiento, examen = cafeteria.iloc[:300], cafeteria.iloc[300:]
X_ent, y_ent = entrenamiento[caracteristicas], entrenamiento["demanda"]
X_exa, y_exa = examen[caracteristicas], examen["demanda"]
print(f"Entrenamiento: {len(X_ent)} días · Examen: {len(X_exa)} días")

---
## 5. 📖 Etapa 5 — Primero, una línea base; después, el modelo

Una **línea base** es la estrategia más simple razonable. Sirve para responder: *"¿el modelo aporta algo?"*. Probaremos dos:
- **Línea base A:** lo que hacen hoy, hornear siempre 120.
- **Línea base B:** el promedio de cada día de la semana (sin ML, solo una tabla).

In [ ]:
def error_medio(prediccion, real):
    return np.mean(np.abs(np.asarray(prediccion) - np.asarray(real)))

pred_A = np.full(len(y_exa), 120)
promedio_por_dia = entrenamiento.groupby("dia_semana")["demanda"].mean()
pred_B = X_exa["dia_semana"].map(promedio_por_dia)

print(f"Línea base A (siempre 120)        : error medio = {error_medio(pred_A, y_exa):.1f} croissants")
print(f"Línea base B (promedio por día)   : error medio = {error_medio(pred_B, y_exa):.1f} croissants")

### 🔮 Ejercicio 5 — ¿Le ganará el modelo a la línea base B?
Ahora entrenamos un **árbol de decisión** (Lección 5.4) que, además del día de la semana, usa la temperatura, la lluvia y los festivos. ¿Cuánto crees que bajará el error respecto a la línea base B? ¿Mucho, poco o nada?

✍️ *Mi predicción:*

In [ ]:
PROFUNDIDAD = 6       # elegida con validación cruzada (Lección 4.6) · ✏️ prueba también 2 y 15
modelo = DecisionTreeRegressor(max_depth=PROFUNDIDAD, random_state=0).fit(X_ent, y_ent)
pred_modelo = modelo.predict(X_exa)
print(f"Árbol (max_depth={PROFUNDIDAD})           : error medio = {error_medio(pred_modelo, y_exa):.1f} croissants")

<details><summary><b>👀 Ver respuesta</b></summary>

Con `max_depth=6` el error baja a **8.4 croissants**, frente a **10.2** de la línea base B y **14.3** de hornear siempre 120. El modelo aprovecha información que la tabla de promedios ignora (frío, lluvia, festivos). Mejora, pero no es magia: hay un ruido de día a día que nadie puede predecir.

✏️ Sorpresa: con `max_depth=2` el error en el examen es **7.6**, ¡mejor que con 6! Y con `max_depth=15`, **8.7**. ¿Entonces elegimos 2? **No.** El 6 se eligió con **validación cruzada** sobre los 300 días de entrenamiento (Lección 4.6), donde funcionaba mejor que 2. El examen son solo 65 días de otoño, y que el 2 gane ahí es en parte **suerte**. Si cambias de modelo cada vez que miras el examen, el examen deja de ser un examen honesto: es la trampa más común del ML.
</details>

---
## 6. 📖 Etapa 6 — Evaluar… en el idioma del negocio

A la dueña no le dice nada "8.4 croissants de error medio". Le importa el **dinero**. Supongamos:
- Cada croissant que **sobra** cuesta **0.40 $** (ingredientes y trabajo tirados).
- Cada croissant que **falta** cuesta **1.20 $** (la ganancia que no se hizo, y un cliente molesto).

In [ ]:
COSTO_SOBRA, COSTO_FALTA = 0.40, 1.20

def dinero_perdido_por_dia(horneados, demanda):
    horneados, demanda = np.round(np.asarray(horneados)), np.asarray(demanda)
    sobran = np.clip(horneados - demanda, 0, None)
    faltan = np.clip(demanda - horneados, 0, None)
    return np.mean(sobran * COSTO_SOBRA + faltan * COSTO_FALTA)

for nombre, pred in [("A · siempre 120", pred_A), ("B · promedio por día", pred_B), ("Árbol", pred_modelo)]:
    print(f"{nombre:<22}: pierde {dinero_perdido_por_dia(pred, y_exa):5.2f} $ por día")

### ✋ Ejercicio 6 — El informe para la dueña
1. ¿Cuánto **ahorraría al año** "El Grano" usando el árbol en lugar de hornear siempre 120? (Calcúlalo con la diferencia por día × 365.)
2. Faltar cuesta 3 veces más que sobrar. ¿Te convendría hornear **exactamente** lo que predice el modelo, o un poco **más**? (Pista: piensa en qué error es más caro.)

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Siempre 120 pierde **10.38 $/día**; el árbol, **6.44 $/día**. La diferencia, ≈ 3.9 $/día × 365 ≈ **1,440 $ al año**. Esa es la cifra que le importa a la dueña, no el "error medio".

   ¿Viste lo raro? La línea base B (promedio por día) se equivoca **menos croissants** que A (10.2 contra 14.3)… pero **pierde más dinero** (11.29 $). Al usar el promedio, se queda corta más o menos la mitad de los días, y quedarse corto es lo caro. En cambio, 120 está por encima de la demanda media (111) y casi siempre se pasa, que es lo barato. **La métrica técnica y la de negocio no siempre coinciden.**
2. Un poco **más**: como quedarse corto cuesta más que pasarse, conviene equivocarse "hacia arriba". Es la misma idea que el umbral de alertas de la Lección 1.4: **la decisión debe reflejar el costo real de cada error** (Lección 4.8). Lo comprobarás en la práctica integradora (11.3).
</details>

---
## 7. 📖 Etapa 7 — Poner el modelo en producción

"Poner en producción" (*deploy*) significa que el modelo **se usa de verdad**, todos los días, por personas o sistemas que no son quien lo construyó. Puede ser algo tan sencillo como una función que la dueña consulta cada noche:

In [ ]:
def cuantos_hornear(dia_semana, temperatura, lluvia, festivo, margen=1.05):
    """Devuelve cuántos croissants hornear mañana (con un pequeño margen de seguridad)."""
    datos = pd.DataFrame([[dia_semana, temperatura, lluvia, festivo]], columns=caracteristicas)
    return int(np.ceil(modelo.predict(datos)[0] * margen))

print("Sábado, 8 °C, sin lluvia :", cuantos_hornear(5, 8, 0, 0))
print("Martes, 27 °C, con lluvia:", cuantos_hornear(1, 27, 1, 0))

### ✏️ Ejercicio 7
Usa `cuantos_hornear` para planificar **mañana**: elige un día, la temperatura prevista y si lloverá o será festivo. Luego responde: ¿qué cosas tendrían que ocurrir **cada noche**, en la vida real, para que esta función se use bien?

In [ ]:
print(cuantos_hornear(dia_semana=4, temperatura=15, lluvia=0, festivo=0))   # ✏️ tu día

<details><summary><b>👀 Ver respuesta</b></summary>

Tendrían que ocurrir, al menos: alguien (o un sistema) **obtiene el pronóstico** del tiempo y el calendario de festivos; **llama** a la función con esos datos; el resultado **llega** a quien hornea; y al día siguiente se **registra** la demanda real para poder vigilar el modelo y reentrenarlo. Poner en producción es construir ese **circuito completo**, no solo guardar el modelo (Lección 15.1).
</details>

---
## 8. 📖 Etapa 8 — Monitorear: los modelos se degradan

El mundo cambia y el modelo no se entera. Imagina que, al mes siguiente, abren unas **oficinas** junto a la cafetería y la demanda sube unos 25 croissants al día. El modelo sigue prediciendo como antes…

In [ ]:
rng_nuevo = np.random.default_rng(99)
mes_nuevo = cafeteria.iloc[:30].copy()                                  # mismas fechas del calendario, un año después
mes_nuevo["temperatura"] = mes_nuevo.temperatura + rng_nuevo.normal(0, 2, 30).round()
mes_nuevo["demanda"] = np.round(100 + extra_por_dia[mes_nuevo.dia_semana] - 1.2 * (mes_nuevo.temperatura - 20)
                                - 10 * mes_nuevo.lluvia + 25 * mes_nuevo.festivo + rng_nuevo.normal(0, 8, 30)
                                + 25).astype(int)                       # ← las oficinas nuevas: +25 cada día

pred_mes = modelo.predict(mes_nuevo[caracteristicas])
print(f"Error medio en el examen original: {error_medio(pred_modelo, y_exa):.1f} croissants")
print(f"Error medio en el mes nuevo      : {error_medio(pred_mes, mes_nuevo.demanda):.1f} croissants")

plt.plot(mes_nuevo.demanda.values, "o-", label="demanda real")
plt.plot(pred_mes, "s--", label="predicción del modelo")
plt.title("El mundo cambió… y el modelo no"); plt.xlabel("día del mes"); plt.legend(); plt.show()

### 🔮 Ejercicio 8 — ¿Cómo lo habrías detectado?
1. Antes de mirar la salida: ¿qué crees que le pasó al error?
2. Si no supieras lo de las oficinas, ¿qué **señal** te habría alertado? ¿Qué harías?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. El error se dispara: de **8.4** a **25.6** croissants. El modelo se queda corto casi todos los días (la línea de predicción va por debajo de la real).
2. La señal: el error **sube** de forma sostenida y siempre en la **misma dirección** (faltan croissants todos los días). Por eso hay que **registrar la demanda real** y comparar con lo predicho cada semana. Qué hacer: **reentrenar** con los datos recientes (y, si el cambio es grande, replantear las características). A este fenómeno se le llama **deriva** (*drift*, Lección 15.2), y es la razón por la que el ciclo de vida es un **ciclo**.
</details>

---
## 9. 🚨 Errores comunes a lo largo del ciclo

| 🚨 Error | Etapa | Consecuencia |
|---|---|---|
| Empezar por el modelo ("usemos redes neuronales") | 1–2 | Se resuelve un problema que nadie tenía |
| No preguntar de dónde sale la etiqueta | 2–3 | Etiquetas engañosas (demanda censurada) |
| Usar características que no se conocerán al predecir | 2–4 | Modelo perfecto en pruebas e inútil en la realidad (fuga de datos, Lección 2.6) |
| Mezclar pasado y futuro al dividir datos temporales | 4 | Evaluación demasiado optimista |
| No comparar con una línea base | 5 | No sabes si el modelo aporta algo |
| Reportar solo métricas técnicas | 6 | El negocio no entiende ni confía en el resultado |
| "Ya está en producción, terminamos" | 8 | El modelo se degrada en silencio |

## 10. 🧠 Resumen en 5 puntos
1. Un proyecto de ML es un **ciclo de 8 etapas**; entrenar el modelo es solo una de ellas.
2. Empieza por la **decisión de negocio** y cómo se mide el éxito; luego tradúcelo a tipo de aprendizaje, tipo de problema, etiqueta y métrica.
3. **Explora** los datos y divide respetando el tiempo: entrena con el pasado, examina con el futuro.
4. Compara siempre con una **línea base** y traduce el resultado a **dinero** o al idioma del negocio.
5. En producción el mundo cambia: **monitorea** y **reentrena**.

---
## 11. 🏋️ Ejercicios finales

### 11.1 Repaso rápido (opción múltiple)

**1.** ¿Cuál debería ser la **primera** etapa de un proyecto de ML?
- a) Elegir el algoritmo · b) Definir la pregunta de negocio y cómo se medirá el éxito · c) Comprar GPUs · d) Poner el modelo en producción

**2.** ¿Para qué sirve una línea base?
- a) Para decorar el informe · b) Para saber si el modelo aporta algo frente a una estrategia simple · c) Para entrenar más rápido · d) Para reemplazar la métrica

**3.** Con datos ordenados en el tiempo, ¿cómo conviene separar el examen?
- a) Al azar · b) Con los datos más antiguos · c) Con los datos más recientes · d) No hace falta examen

**4.** Un modelo funcionaba bien y, tras unos meses, su error sube cada semana. ¿Qué es lo más probable?
- a) El modelo "se cansó" · b) Los datos del mundo cambiaron (deriva) · c) Python se actualizó · d) Es normal y no hay que hacer nada

**5.** ¿Cuál de estas características **no** podría usar el modelo de "El Grano" para decidir cuánto hornear?
- a) Día de la semana · b) Temperatura prevista · c) Croissants vendidos ese mismo día · d) Si es festivo

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** · 2. **b)** · 3. **c)** · 4. **b)** · 5. **c)** Se decide **antes** de abrir: las ventas de ese día aún no existen (sería fuga de datos).
</details>

### 11.2 Preguntas para pensar (sin ejecutar)

**A.** Un hospital quiere predecir qué pacientes volverán a ingresar en los 30 días siguientes al alta. Recorre las **8 etapas** del ciclo con una frase cada una.

**B.** ¿Por qué crees que tantos proyectos de ML "funcionan en el notebook" pero nunca llegan a usarse? Da al menos dos razones relacionadas con las etapas del ciclo.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

**A.** Un ejemplo:
1. **Pregunta:** ¿a qué pacientes damos seguimiento telefónico tras el alta para evitar reingresos?
2. **Planteamiento:** supervisado, clasificación binaria (¿reingresa en 30 días?), con probabilidad; métrica: cuántos reingresos detectamos entre los que llamamos.
3. **Datos:** historiales de altas de los últimos años, con la etiqueta "reingresó o no".
4. **Explorar/preparar:** revisar faltantes, usar solo información disponible **en el momento del alta**, dividir por fecha.
5. **Línea base y modelo:** línea base = llamar a los pacientes de más edad; luego un modelo.
6. **Evaluar:** ¿cuántos reingresos evitaríamos con las mismas llamadas? ¿Es justo con todos los grupos de pacientes?
7. **Producción:** cada día, una lista priorizada de pacientes para el equipo de enfermería.
8. **Monitoreo:** comparar mes a mes los reingresos reales con los predichos; reentrenar si cambian los protocolos.

**B.** Por ejemplo: (1) se empezó por el modelo y no por una **decisión** concreta, así que nadie sabe qué hacer con la predicción; (2) se usaron características que **no existen** en el momento de usar el modelo; (3) los resultados se presentaron como "92 % de exactitud" y no en el idioma del negocio, así que nadie confió; (4) nadie se encargó de construir el circuito de **producción** ni de **monitorear**.
</details>

### 11.3 🧩 Práctica integradora: el margen de seguridad óptimo

En el Ejercicio 6 viste que faltar cuesta más que sobrar. Vas a encontrar el **margen** que minimiza el dinero perdido de "El Grano". 4 pasos cortos; revisa cada respuesta antes de seguir.

**Paso 1 🔮 — Tu hipótesis.** Si hornearas un 5 %, 10 % o 20 % más de lo que predice el modelo, ¿qué margen crees que perdería **menos** dinero?

✍️ *Mi hipótesis:*

**Paso 2 ▶️ — El experimento (con el entrenamiento, ¡no con el examen!).** Para no hacer trampa, elegimos el margen con los **últimos 60 días del entrenamiento**, usando un modelo entrenado con los 240 días anteriores.

In [ ]:
sub_ent, validacion = entrenamiento.iloc[:240], entrenamiento.iloc[240:]
modelo_val = DecisionTreeRegressor(max_depth=6, random_state=0).fit(sub_ent[caracteristicas], sub_ent["demanda"])
pred_val = modelo_val.predict(validacion[caracteristicas])

margenes = np.arange(0.90, 1.31, 0.05)
perdidas = [dinero_perdido_por_dia(pred_val * m, validacion["demanda"]) for m in margenes]
for m, p in zip(margenes, perdidas):
    print(f"margen {m:.2f} → pierde {p:5.2f} $/día " + "█" * int(p * 2))

mejor_margen = margenes[int(np.argmin(perdidas))]
print(f"\nMejor margen en validación: {mejor_margen:.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

El mejor margen es **1.05** (hornear un 5 % más de lo predicho): pierde 4.67 $/día, con 1.10 muy cerca (4.88). Con menos margen faltan croissants (caro); con mucho más, sobran demasiados. Es la curva en "U" que verás muchas veces en el curso.

Fíjate en el detalle metodológico: el margen es un **hiperparámetro de la decisión**, y lo elegimos con datos de **validación**, no con el examen (Lección 2.5).
</details>

**Paso 3 ▶️ — El examen final, una sola vez.** Ahora sí, aplicamos el margen elegido al modelo original en los 65 días de examen.

In [ ]:
print(f"Árbol sin margen        : pierde {dinero_perdido_por_dia(pred_modelo, y_exa):.2f} $/día")
print(f"Árbol con margen {mejor_margen:.2f}  : pierde {dinero_perdido_por_dia(pred_modelo * mejor_margen, y_exa):.2f} $/día")
print(f"Siempre 120             : pierde {dinero_perdido_por_dia(pred_A, y_exa):.2f} $/día")

<details><summary><b>👀 Ver respuesta</b></summary>

Con el margen, la pérdida en el examen baja de **6.44** a **5.62 $/día**, frente a **10.38** de hornear siempre 120. Pasar de "siempre 120" a "árbol + margen" ahorra unos **4.8 $/día ≈ 1,700 $ al año**, casi la mitad de lo que se perdía.

Y lo más importante: el margen salió de la **regla de negocio** (faltar cuesta 3 veces más que sobrar) y se eligió **sin mirar el examen**, así que el resultado es honesto.
</details>

**Paso 4 ✋ — Tu informe en 4 líneas.** Escribe para la dueña de "El Grano": qué recomiendas, cuánto ahorrará, qué necesitas que hagan cada noche y qué vigilarás cada semana.

✍️ *Mi informe:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo:
1. **Recomendación:** decidir cada noche cuántos croissants hornear con el modelo (día de la semana, pronóstico de temperatura y lluvia, festivos), horneando un 5 % más de lo que predice.
2. **Ahorro:** de ~10.4 $ a ~5.6 $ perdidos por día: unos **1,700 $ al año**, casi la mitad de lo que hoy se tira o se deja de vender.
3. **Cada noche:** anotar el pronóstico, consultar el modelo y pasar la cifra a quien hornea; **registrar la demanda real** (incluidos los clientes que se quedaron sin croissant).
4. **Cada semana:** comparar lo predicho con lo vendido. Si el error sube de forma sostenida (como con las oficinas nuevas), reentrenar con los datos recientes.
</details>

---
## 📚 Para profundizar
- 📘 **Lectura:** Google, *Rules of Machine Learning* (Martin Zinkevich): 43 reglas prácticas; las primeras dicen "no tengas miedo de lanzar un producto sin ML".
- 📘 **Lectura:** el modelo **CRISP-DM**, el ciclo de vida clásico de los proyectos de datos (busca "CRISP-DM").
- 📺 **Video:** Andrew Ng, *Machine Learning Yearning* / cursos de *MLOps* de DeepLearning.AI (inglés, con subtítulos).

---
⬅️ **Anterior:** [1.4 Tipos de problema: regresión, clasificación, clustering, ranking y generación](1.4-tipos-de-problema.ipynb) · ➡️ **Siguiente:** [1.6 "No hay almuerzo gratis": por qué no existe un modelo perfecto](1.6-no-hay-almuerzo-gratis.ipynb)